# Fine-tune AraBERT v02 for 3-class Arabic sentiment (Colab)

GPU wrapper around `training/train.py` — the script is the source of truth; this notebook only sets up
a GPU runtime, runs it, and packages the weights for download.

**Runtime → Change runtime type → T4 GPU.** A full run (3 epochs on ASTD, ~6.8k tweets) should take a few minutes on a T4 (estimate; on a 12-core CPU it is ~30 s per batch of 32, i.e. hours).

Output: `models/arabert_sentiment/` (config, safetensors, tokenizer, `training_meta.json` with test metrics),
zipped for download. Unzip it into the same path in the project and `consumer_prediction` picks it up —
even while running (hot reload).

> Background: `NLP_for_Sentiment_Analysis.ipynb` (project root) is the original exploration — binary, TF/Keras,
> AraBERT v1. This pipeline moved to PyTorch + v02 + three classes; see the docstring at the top of `train.py`.

## 1. Get the project
Either clone your repo (set `REPO_URL`), or upload a zip of the project folder and unzip it to `/content/Taagger`.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<you>/Taagger.git"; leave empty if uploading a zip instead
import os
if REPO_URL and not os.path.exists("/content/Taagger"):
    !git clone --depth 1 {REPO_URL} /content/Taagger
%cd /content/Taagger
!ls

## 2. Install training dependencies
Colab already ships a CUDA build of torch; this adds the rest.

In [ ]:
!pip install -q "transformers==5.18.0" "arabert==1.0.1" pandas pyarrow scikit-learn
import torch; print(torch.__version__, "CUDA:", torch.cuda.is_available())

## 3. Download ASTD and write stratified splits
`Objective` tweets are folded into `neutral` (use `--objective drop` to remove them instead).

In [ ]:
!python scripts/prepare_dataset.py

## 4. Fine-tune
Best epoch by validation macro-F1 is kept; test metrics are printed at the end and stored in `training_meta.json`.

In [ ]:
!python training/train.py --epochs 3 --batch-size 32 --lr 2e-5

In [ ]:
import json
meta = json.load(open("models/arabert_sentiment/training_meta.json"))
print(meta["model_version"])
print("test accuracy", meta["test"]["accuracy"], "| macro-F1", meta["test"]["macro_f1"])
for h in meta["history"]: print(h)

## 5. Package the weights
Download the zip, then on your machine: unzip into `models/arabert_sentiment/` at the project root.

In [ ]:
!cd models && zip -qr ../arabert_sentiment.zip arabert_sentiment && cd .. && ls -lh arabert_sentiment.zip
from google.colab import files
files.download("arabert_sentiment.zip")